In [1]:
# 05_additional_experiments.ipynb
# Additional experiments for the manuscript (single cell). Study 1: C5 leakage-free re-estimation,
# C2 economic significance (IDI/NRI/decision value), C6 all-pairs indicator distribution,
# C7/C10 asymmetry Wald test + imbalance robustness. Study 2: C8 QWK bootstrap CI + RF
# hyperparameters, C9 proportional-odds validation vs statsmodels. Plus the all-pairs figure.
# Paths are relative to notebooks/. Run top-to-bottom.

# ===== additional experiments: Study 1 =====
# Additional experiments, Study 1
# Supplementary analyses: leakage-free re-estimation, economic significance (IDI/NRI/decision value),
# all-pairs indicator distribution, asymmetry (Wald) test, and class-imbalance robustness.
import os, json, warnings, itertools
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from scipy.io import arff
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, brier_score_loss

DATA = os.environ.get("DATA", "../data")
OUT  = os.environ.get("OUT", "../results")
os.makedirs(f"{OUT}/tables", exist_ok=True)

def load_year(n):
    d, _ = arff.loadarff(f"{DATA}/polish/{n}year.arff"); df = pd.DataFrame(d)
    cls = [c for c in df.columns if c.lower() == "class"][0]
    df[cls] = df[cls].apply(lambda b: int(b.decode()) if isinstance(b, bytes) else int(b))
    fe = [c for c in df.columns if c != cls]
    df.columns = [f"Attr{i+1}" for i in range(len(fe))] + ["target"]; df["horizon"] = n
    return df
pooled = pd.concat([load_year(n) for n in range(1, 6)], ignore_index=True)
feats = [c for c in pooled.columns if c.startswith("Attr")]
y = pooled["target"].astype(int).values
imp = SimpleImputer(strategy="median")
X = pd.DataFrame(imp.fit_transform(pooled[feats]), columns=feats)
Hd = pd.get_dummies(pooled["horizon"], prefix="h", drop_first=True).astype(float).reset_index(drop=True)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
proxies = ["Attr1", "Attr2"]  # profitability, leverage

def delong(y_true, p1, p2):
    y_true = np.asarray(y_true); pos = y_true == 1; neg = ~pos; m = pos.sum(); n = neg.sum()
    def st(p):
        xp, xn = p[pos], p[neg]
        V10 = np.array([(np.sum(xn < xi) + 0.5*np.sum(xn == xi))/n for xi in xp])
        V01 = np.array([(np.sum(xp > xj) + 0.5*np.sum(xp == xj))/m for xj in xn])
        return V10.mean(), V10, V01
    a1, v10a, v01a = st(p1); a2, v10b, v01b = st(p2)
    S10 = np.cov(np.vstack([v10a, v10b])); S01 = np.cov(np.vstack([v01a, v01b]))
    var = S10/m + S01/n; v = var[0, 0] + var[1, 1] - 2*var[0, 1]
    z = (a1 - a2)/np.sqrt(v) if v > 0 else 0.0
    return float(a1), float(a2), float(z), float(2*(1 - stats.norm.cdf(abs(z))))

import statsmodels.api as sm
out = {}

# ================= C5: leakage-free nested estimation =================
# Base grade bucket edges computed on TRAIN fold only; S and grade predicted out-of-fold.
base_feats = [c for c in feats if c not in proxies]
p0 = np.empty(len(y)); p1 = np.empty(len(y)); Goof = np.empty(len(y)); Soof = np.empty(len(y))
for tr, te in cv.split(X, y):
    sc_b = StandardScaler().fit(X.iloc[tr][base_feats])
    base = LogisticRegression(max_iter=3000, class_weight="balanced").fit(sc_b.transform(X.iloc[tr][base_feats]), y[tr])
    pd_tr = base.predict_proba(sc_b.transform(X.iloc[tr][base_feats]))[:, 1]
    pd_te = base.predict_proba(sc_b.transform(X.iloc[te][base_feats]))[:, 1]
    edges = np.quantile(pd_tr, np.linspace(0, 1, 8)[1:-1])           # edges from TRAIN only
    G_tr = np.digitize(pd_tr, edges); G_te = np.digitize(pd_te, edges)
    sc_s = StandardScaler().fit(X.iloc[tr][proxies])
    sm_s = LogisticRegression(max_iter=3000, class_weight="balanced").fit(sc_s.transform(X.iloc[tr][proxies]), y[tr])
    S_tr = sm_s.predict_proba(sc_s.transform(X.iloc[tr][proxies]))[:, 1]
    S_te = sm_s.predict_proba(sc_s.transform(X.iloc[te][proxies]))[:, 1]
    Htr, Hte = Hd.iloc[tr].values, Hd.iloc[te].values
    M0tr = np.column_stack([G_tr, Htr]); M0te = np.column_stack([G_te, Hte])
    M1tr = np.column_stack([G_tr, S_tr, Htr]); M1te = np.column_stack([G_te, S_te, Hte])
    m0 = LogisticRegression(max_iter=3000).fit(M0tr, y[tr]); m1 = LogisticRegression(max_iter=3000).fit(M1tr, y[tr])
    p0[te] = m0.predict_proba(M0te)[:, 1]; p1[te] = m1.predict_proba(M1te)[:, 1]
    Goof[te] = G_te; Soof[te] = S_te
auc0, auc1 = roc_auc_score(y, p0), roc_auc_score(y, p1)
a1d, a0d, zz, pv = delong(y, p1, p0)
out["C5_leakage_free"] = {
    "note": "Bucket edges from TRAIN fold only; grade, signal, and M0/M1 all fit in-fold (fully nested OOF).",
    "AUC_M0": round(auc0, 4), "AUC_M1": round(auc1, 4),
    "AR_M0": round(2*auc0-1, 4), "AR_M1": round(2*auc1-1, 4),
    "DeLong_z": round(zz, 2), "DeLong_p": f"{pv:.2e}",
    "Brier_M0": round(brier_score_loss(y, p0), 5), "Brier_M1": round(brier_score_loss(y, p1), 5),
}

# ================= C2: economic significance =================
# IDI and continuous NRI (Pencina), overall and within middle band; Brier skill.
ev, nv = y == 1, y == 0
IDI = (p1[ev].mean() - p0[ev].mean()) - (p1[nv].mean() - p0[nv].mean())
# continuous NRI: fraction of events with p1>p0 minus p1<p0, plus same (reversed) for non-events
nri_ev = np.mean(p1[ev] > p0[ev]) - np.mean(p1[ev] < p0[ev])
nri_ne = np.mean(p1[nv] < p0[nv]) - np.mean(p1[nv] > p0[nv])
NRI = nri_ev + nri_ne
brier_skill = 1 - brier_score_loss(y, p1) / brier_score_loss(y, p0)
mid = np.isin(Goof, [2, 3, 4])
evm, nvm = (y == 1) & mid, (y == 0) & mid
IDI_mid = (p1[evm].mean() - p0[evm].mean()) - (p1[nvm].mean() - p0[nvm].mean())
NRI_mid = (np.mean(p1[evm] > p0[evm]) - np.mean(p1[evm] < p0[evm])) + (np.mean(p1[nvm] < p0[nvm]) - np.mean(p1[nvm] > p0[nvm]))
# decision value: expected misclassification cost at cost ratio c (FN:FP), optimal threshold each model
def min_cost(p, c):
    thr = np.unique(np.quantile(p, np.linspace(0.01, 0.99, 99)))
    best = np.inf
    for t in thr:
        fp = np.sum((p >= t) & (y == 0)); fn = np.sum((p < t) & (y == 1))
        best = min(best, c*fn + fp)
    return best
val = {}
for c in [5, 10, 20]:
    c0, c1 = min_cost(p0, c), min_cost(p1, c)
    val[f"cost_ratio_{c}"] = {"min_cost_M0": int(c0), "min_cost_M1": int(c1), "reduction_pct": round(100*(c0-c1)/c0, 2)}
out["C2_economic_significance"] = {
    "IDI": round(float(IDI), 5), "NRI_continuous": round(float(NRI), 4), "Brier_skill_score": round(float(brier_skill), 4),
    "IDI_mid_band": round(float(IDI_mid), 5), "NRI_mid_band": round(float(NRI_mid), 4),
    "decision_value": val,
}

# ================= C7/C10: asymmetry formal test + subsample sizes =================
D = pd.DataFrame({"y": y, "G": Goof.astype(float), "S": Soof}).reset_index(drop=True)
D["dev"] = D["S"] - D.groupby("G")["S"].transform("mean")
D["det"] = D["dev"].clip(lower=0); D["imp"] = D["dev"].clip(upper=0)
Xh3 = sm.add_constant(pd.concat([D[["G", "det", "imp"]], Hd], axis=1).astype(float))
m3 = sm.Logit(D["y"], Xh3).fit(disp=0)
# Wald test of |gamma+| = |gamma-|  <=>  det + imp = 0
wald = m3.t_test("det + imp = 0")
out["C10_asymmetry_test"] = {
    "coef_det": round(float(m3.params["det"]), 3), "z_det": round(float(m3.tvalues["det"]), 2),
    "coef_imp": round(float(m3.params["imp"]), 3), "z_imp": round(float(m3.tvalues["imp"]), 2),
    "n_deterioration": int((D["dev"] > 0).sum()), "n_improvement": int((D["dev"] < 0).sum()),
    "Wald_det_plus_imp_eq_0_stat": round(float(wald.statistic.squeeze()), 2),
    "Wald_p": f"{float(wald.pvalue):.2e}",
}

# ================= C7: imbalance robustness (AUC/AR of M1 under treatments) =================
from sklearn.utils import resample
rows = []
for name, cw in [("none", None), ("balanced", "balanced")]:
    pp = np.empty(len(y))
    for tr, te in cv.split(X, y):
        sc = StandardScaler().fit(X.iloc[tr][base_feats]);
        bpd_tr = LogisticRegression(max_iter=3000, class_weight=cw).fit(sc.transform(X.iloc[tr][base_feats]), y[tr])
        g_tr = bpd_tr.predict_proba(sc.transform(X.iloc[tr][base_feats]))[:,1]; g_te = bpd_tr.predict_proba(sc.transform(X.iloc[te][base_feats]))[:,1]
        ed = np.quantile(g_tr, np.linspace(0,1,8)[1:-1]); G_te = np.digitize(g_te, ed); G_tr = np.digitize(g_tr, ed)
        scs = StandardScaler().fit(X.iloc[tr][proxies]); s_mod = LogisticRegression(max_iter=3000, class_weight=cw).fit(scs.transform(X.iloc[tr][proxies]), y[tr])
        s_tr = s_mod.predict_proba(scs.transform(X.iloc[tr][proxies]))[:,1]; s_te = s_mod.predict_proba(scs.transform(X.iloc[te][proxies]))[:,1]
        mm = LogisticRegression(max_iter=3000, class_weight=cw).fit(np.column_stack([G_tr, s_tr, Hd.iloc[tr].values]), y[tr])
        pp[te] = mm.predict_proba(np.column_stack([G_te, s_te, Hd.iloc[te].values]))[:,1]
    rows.append({"treatment": name, "AUC_M1": round(roc_auc_score(y, pp), 4), "AR_M1": round(2*roc_auc_score(y, pp)-1, 4)})
# oversample events in training
pp = np.empty(len(y))
for tr, te in cv.split(X, y):
    tr_idx = list(tr); ev_tr = [i for i in tr if y[i]==1]
    ev_up = resample(ev_tr, replace=True, n_samples=sum(y[tr]==0)-len(ev_tr), random_state=0)
    idx = np.array(tr_idx + list(ev_up))
    sc = StandardScaler().fit(X.iloc[idx][base_feats]); b = LogisticRegression(max_iter=3000).fit(sc.transform(X.iloc[idx][base_feats]), y[idx])
    g_tr = b.predict_proba(sc.transform(X.iloc[idx][base_feats]))[:,1]; g_te = b.predict_proba(sc.transform(X.iloc[te][base_feats]))[:,1]
    ed = np.quantile(g_tr, np.linspace(0,1,8)[1:-1]); G_te = np.digitize(g_te, ed); G_tr = np.digitize(g_tr, ed)
    scs = StandardScaler().fit(X.iloc[idx][proxies]); s = LogisticRegression(max_iter=3000).fit(scs.transform(X.iloc[idx][proxies]), y[idx])
    s_tr = s.predict_proba(scs.transform(X.iloc[idx][proxies]))[:,1]; s_te = s.predict_proba(scs.transform(X.iloc[te][proxies]))[:,1]
    mm = LogisticRegression(max_iter=3000).fit(np.column_stack([G_tr, s_tr, Hd.iloc[idx].values]), y[idx])
    pp[te] = mm.predict_proba(np.column_stack([G_te, s_te, Hd.iloc[te].values]))[:,1]
rows.append({"treatment": "oversample", "AUC_M1": round(roc_auc_score(y, pp), 4), "AR_M1": round(2*roc_auc_score(y, pp)-1, 4)})
out["C7_imbalance_robustness"] = rows

# ================= C6: all-pairs incremental LR across indicator categories =================
# Curated 12 ratios across four categories (Zieba et al. definitions).
cats = {
    "leverage":      ["Attr2", "Attr10", "Attr17"],   # liab/assets, equity/assets, assets/liab
    "profitability": ["Attr1", "Attr7", "Attr18"],    # ROA(net), EBIT/assets, gross/assets
    "liquidity":     ["Attr4", "Attr46", "Attr50"],   # current, quick, current assets/liab
    "activity":      ["Attr9", "Attr36", "Attr60"],   # sales/assets x2, sales/inventory
}
ratio2cat = {a: c for c, L in cats.items() for a in L}
cand = list(ratio2cat.keys())
# fixed full-64 base grade (OOF, within-fold edges) reused for every pair
def oof_pd(cols, cw="balanced"):
    pdx = np.empty(len(y))
    for tr, te in cv.split(X, y):
        sc = StandardScaler().fit(X.iloc[tr][cols]); mdl = LogisticRegression(max_iter=2000, class_weight=cw).fit(sc.transform(X.iloc[tr][cols]), y[tr])
        pdx[te] = mdl.predict_proba(sc.transform(X.iloc[te][cols]))[:, 1]
    return pdx
Gfull = pd.qcut(pd.Series(oof_pd(feats)), 7, labels=False, duplicates="drop").values.astype(float)
base_df = pd.concat([pd.Series(Gfull, name="G"), Hd], axis=1).astype(float)
m_base = sm.Logit(y, sm.add_constant(base_df)).fit(disp=0)
pair_rows = []
for a, b in itertools.combinations(cand, 2):
    s = oof_pd([a, b])
    d = sm.add_constant(pd.concat([base_df, pd.Series(s, name="S")], axis=1).astype(float))
    m = sm.Logit(y, d).fit(disp=0)
    lr = 2*(m.llf - m_base.llf)
    kinds = tuple(sorted([ratio2cat[a], ratio2cat[b]]))
    pair_rows.append({"a": a, "b": b, "cat_a": ratio2cat[a], "cat_b": ratio2cat[b],
                      "pair_type": "+".join(kinds), "incremental_LR": round(float(lr), 1)})
pairs = pd.DataFrame(pair_rows).sort_values("incremental_LR", ascending=False).reset_index(drop=True)
pairs["rank"] = pairs.index + 1
levprof = pairs[pairs["pair_type"] == "leverage+profitability"]
out["C6_all_pairs"] = {
    "note": "Incremental LR of adding each 2-ratio signal over a fixed full-model grade; 12 curated ratios, 66 pairs.",
    "n_pairs": len(pairs),
    "levprof_pairs_mean_LR": round(float(levprof["incremental_LR"].mean()), 1),
    "levprof_best_rank": int(levprof["rank"].min()), "levprof_worst_rank": int(levprof["rank"].max()),
    "overall_median_LR": round(float(pairs["incremental_LR"].median()), 1),
    "top5": pairs.head(5)[["a", "b", "pair_type", "incremental_LR"]].to_dict("records"),
    "by_type_mean": pairs.groupby("pair_type")["incremental_LR"].mean().round(1).sort_values(ascending=False).to_dict(),
}
pairs.to_csv(f"{OUT}/tables/review_all_pairs_LR.csv", index=False)

js = json.dumps(out, ensure_ascii=False, indent=2)
print(js)
open(f"{OUT}/tables/review_study1.json", "w").write(js)


# ===== additional experiments: Study 2 =====
# Additional experiments, Study 2
# Rating-recovery robustness: random-forest hyperparameters disclosed + QWK bootstrap CIs
# (full vs two-proxy), and validation of the L2 proportional-odds model against statsmodels
# OrderedModel on a converging low-dimensional case (coefficient agreement).
import os, json, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import cohen_kappa_score
from scipy.optimize import minimize
from scipy.special import expit
from statsmodels.miscmodels.ordinal_model import OrderedModel

DATA = os.environ.get("DATA", "../data"); OUT = os.environ.get("OUT", "../results")
os.makedirs(f"{OUT}/tables", exist_ok=True)
df = pd.read_csv(f"{DATA}/corporate_rating.csv"); df.columns = [c.strip().replace(" ", "_") for c in df.columns]
meta = ["Rating", "Name", "Symbol", "Rating_Agency_Name", "Date", "Sector"]
feats = [c for c in df.columns if c not in meta]; proxies = ["debtRatio", "returnOnAssets"]
order = ["AAA", "AA", "A", "BBB", "BB", "B", "CCC"]; K = len(order)
df["R"] = df["Rating"].replace({"CC": "CCC", "C": "CCC", "D": "CCC"})
df["y"] = df["R"].map({r: i for i, r in enumerate(order)}); y = df["y"].values
X = df[feats].fillna(df[feats].median())
Xz = pd.DataFrame(StandardScaler().fit_transform(X), columns=feats, index=df.index).clip(-5, 5)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
out = {}

# ---- C8: RF hyperparameters + QWK bootstrap CI ----
RF_PARAMS = {"n_estimators": 300, "max_depth": None, "min_samples_leaf": 1,
             "max_features": "sqrt", "random_state": 0, "n_jobs": -1}
pred_full = cross_val_predict(RandomForestClassifier(**RF_PARAMS), X[feats].values, y, cv=cv)
pred_two  = cross_val_predict(RandomForestClassifier(**RF_PARAMS), X[proxies].values, y, cv=cv)
def qwk_ci(yt, yp, B=2000, seed=0):
    rng = np.random.default_rng(seed); n = len(yt); vals = np.empty(B)
    for b in range(B):
        idx = rng.integers(0, n, n)
        vals[b] = cohen_kappa_score(yt[idx], yp[idx], weights="quadratic")
    return float(np.mean(vals)), float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))
mf, lf, uf = qwk_ci(y, pred_full); mt, lt, ut = qwk_ci(y, pred_two)
out["C8_rf"] = {
    "hyperparameters": RF_PARAMS,
    "QWK_full": round(float(cohen_kappa_score(y, pred_full, weights="quadratic")), 4),
    "QWK_full_CI95": [round(lf, 4), round(uf, 4)],
    "QWK_2proxy": round(float(cohen_kappa_score(y, pred_two, weights="quadratic")), 4),
    "QWK_2proxy_CI95": [round(lt, 4), round(ut, 4)],
    "CIs_disjoint": bool(lf > ut),
}

# ---- C9: validate hand-rolled penalized proportional-odds vs statsmodels (2-proxy case) ----
def fit_po(Xm, yy, lam):
    Xm = np.asarray(Xm, float); n, p = Xm.shape
    def unpack(th):
        a0 = th[0]; dl = th[1:K-1]; beta = th[K-1:]
        return np.concatenate([[a0], a0 + np.cumsum(np.exp(dl))]), beta
    def nll(th):
        cuts, beta = unpack(th); eta = Xm @ beta
        F = expit(cuts[None, :] - eta[:, None]); P = np.empty((n, K))
        P[:, 0] = F[:, 0]; P[:, 1:K-1] = F[:, 1:] - F[:, :-1]; P[:, K-1] = 1 - F[:, -1]
        P = np.clip(P, 1e-12, 1)
        return -np.log(P[np.arange(n), yy]).sum() + lam*np.sum(beta**2)
    r = minimize(nll, np.concatenate([[0.0], np.zeros(K-2), np.zeros(p)]), method="L-BFGS-B", options={"maxiter": 800})
    return unpack(r.x)
# statsmodels reference (no penalty), our model with tiny penalty -> should match closely
sm_mod = OrderedModel(y, Xz[proxies], distr="logit").fit(method="lbfgs", disp=0, maxiter=500)
sm_beta = np.array([sm_mod.params[p] for p in proxies])
_, po_beta = fit_po(Xz[proxies].values, y, lam=1e-4)
out["C9_po_validation"] = {
    "note": "Hand-rolled L2 proportional-odds (lambda=1e-4) vs statsmodels OrderedModel, 2-proxy case.",
    "statsmodels_beta": {p: round(float(b), 4) for p, b in zip(proxies, sm_beta)},
    "handrolled_beta":  {p: round(float(b), 4) for p, b in zip(proxies, po_beta)},
    "max_abs_diff": round(float(np.max(np.abs(sm_beta - po_beta))), 4),
    "sign_and_order_agree": bool(np.all(np.sign(sm_beta) == np.sign(po_beta))),
}

js = json.dumps(out, ensure_ascii=False, indent=2); print(js)
open(f"{OUT}/tables/review_study2.json", "w").write(js)


# ===== additional experiments: figure =====
# Grayscale figure for the manuscript (png + pdf, dpi 600, no captions)
import os, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, seaborn as sns

OUT = os.environ.get("OUT", "../results"); FIG = f"{OUT}/figures"; TAB = f"{OUT}/tables"
os.makedirs(FIG, exist_ok=True)
sns.set_theme(style="white", context="paper")
plt.rcParams.update({"savefig.dpi": 600, "figure.dpi": 110, "font.size": 11,
                     "axes.edgecolor": "black", "axes.linewidth": 0.8, "axes.grid": False})
def save_fig(fig, name):
    fig.tight_layout(); fig.savefig(f"{FIG}/{name}.png", dpi=600, bbox_inches="tight")
    fig.savefig(f"{FIG}/{name}.pdf", bbox_inches="tight"); plt.close(fig)

# ---- Fig 13: all-pairs incremental LR by pair type (Study 1) ----
pairs = pd.read_csv(f"{TAB}/review_all_pairs_LR.csv")
order_types = pairs.groupby("pair_type")["incremental_LR"].median().sort_values(ascending=False).index.tolist()
fig, ax = plt.subplots(figsize=(6.6, 3.6))
greys = {t: ("0.25" if t == "leverage+profitability" else "0.78") for t in order_types}
for i, t in enumerate(order_types):
    vals = pairs.loc[pairs.pair_type == t, "incremental_LR"].values
    ax.scatter(np.full_like(vals, i, dtype=float) + np.random.default_rng(0).uniform(-0.12, 0.12, len(vals)),
               vals, s=28, facecolor=greys[t], edgecolor="black", linewidth=0.5, zorder=3)
    ax.hlines(np.median(vals), i-0.25, i+0.25, color="black", linewidth=1.4, zorder=4)
ax.set_xticks(range(len(order_types)))
ax.set_xticklabels([t.replace("+", " + ") for t in order_types], fontsize=7.5, rotation=35, ha="right")
ax.set_ylabel("Incremental LR over full-model grade")
ax.margins(x=0.03)
ax.axhline(3.84, color="0.5", linestyle="--", linewidth=0.8)  # chi2(1) 5% ~ but LR df varies; reference only
save_fig(fig, "fig13_review_all_pairs_LR")
print("[05c] wrote fig13_review_all_pairs_LR (png+pdf)")


{
  "C5_leakage_free": {
    "note": "Bucket edges from TRAIN fold only; grade, signal, and M0/M1 all fit in-fold (fully nested OOF).",
    "AUC_M0": 0.7358,
    "AUC_M1": 0.7384,
    "AR_M0": 0.4717,
    "AR_M1": 0.4769,
    "DeLong_z": 3.16,
    "DeLong_p": "1.60e-03",
    "Brier_M0": 0.04377,
    "Brier_M1": 0.04359
  },
  "C2_economic_significance": {
    "IDI": 0.00421,
    "NRI_continuous": -0.0635,
    "Brier_skill_score": 0.0042,
    "IDI_mid_band": 0.00059,
    "NRI_mid_band": 0.2133,
    "decision_value": {
      "cost_ratio_5": {
        "min_cost_M0": 9810,
        "min_cost_M1": 9766,
        "reduction_pct": 0.45
      },
      "cost_ratio_10": {
        "min_cost_M0": 17198,
        "min_cost_M1": 17252,
        "reduction_pct": -0.31
      },
      "cost_ratio_20": {
        "min_cost_M0": 26665,
        "min_cost_M1": 26401,
        "reduction_pct": 0.99
      }
    }
  },
  "C10_asymmetry_test": {
    "coef_det": 2.757,
    "z_det": 7.53,
    "coef_imp": 1.187,
    "z

{
  "C8_rf": {
    "hyperparameters": {
      "n_estimators": 300,
      "max_depth": null,
      "min_samples_leaf": 1,
      "max_features": "sqrt",
      "random_state": 0,
      "n_jobs": -1
    },
    "QWK_full": 0.654,
    "QWK_full_CI95": [
      0.6253,
      0.6799
    ],
    "QWK_2proxy": 0.345,
    "QWK_2proxy_CI95": [
      0.3028,
      0.3839
    ],
    "CIs_disjoint": true
  },
  "C9_po_validation": {
    "note": "Hand-rolled L2 proportional-odds (lambda=1e-4) vs statsmodels OrderedModel, 2-proxy case.",
    "statsmodels_beta": {
      "debtRatio": 0.4733,
      "returnOnAssets": -0.6122
    },
    "handrolled_beta": {
      "debtRatio": 0.4734,
      "returnOnAssets": -0.6121
    },
    "max_abs_diff": 0.0001,
    "sign_and_order_agree": true
  }
}


[05c] wrote fig13_review_all_pairs_LR (png+pdf)
